In [ ]:
#The code in this block comes directly from data_sort_and_split.ipynb instead sclaing is uneccesary as it is not required for Forest models
import pandas as pd
from sklearn.model_selection import train_test_split

heart_data = pd.read_csv("heart.csv")

heart_data['Sex_F'] = heart_data['Sex'].map({'M': 0, 'F': 1})
heart_data['ExerciseAngina'] = heart_data['ExerciseAngina'].map({'N': 0, 'Y': 1})
heart_data = heart_data.drop(columns=['Sex'])

heart_data['ChestPainType'] = pd.Categorical(heart_data['ChestPainType'], categories=['ASY', 'ATA', 'NAP', 'TA'])
heart_data['RestingECG'] = pd.Categorical(heart_data['RestingECG'], categories=['Normal', 'LVH', 'ST'])
heart_data['ST_Slope'] = pd.Categorical(heart_data['ST_Slope'], categories=['Up', 'Flat', 'Down'])

categorical_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
heart_data = pd.get_dummies(heart_data, columns=categorical_cols, drop_first=True, dtype=int)

feature_matrix = heart_data.drop("HeartDisease", axis=1)
target_labels = heart_data["HeartDisease"]

features_train, features_test, targets_train, targets_test = train_test_split(
    feature_matrix,
    target_labels,
    test_size=0.20,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Why: n_estimators=100 builds 100 individual decision trees and combines their predictions,
# which produces a much more stable and accurate result than a single decision tree alone.
# Why: criterion='entropy' measures the quality of each split using information gain,
# allowing each tree to always choose the most informative question to ask about a patient.
# Why: max_depth=10 limits how deep each tree can grow, preventing individual trees from
# memorizing the training data and failing on new patients (overfitting).
# Why: max_features='sqrt' means each tree only sees a random subset of features at each
# split, forcing the trees to be different from each other and reducing overfitting.
# Why: bootstrap=True means each tree is trained on a random sample of patients with
# replacement, which adds diversity across trees and improves generalization.
# Why: class_weight='balanced' ensures the model pays extra attention to heart disease
# patients since our dataset may have more healthy patients than sick ones.
# Why: random_state=42 ensures the model produces the same results every time it is run,
# making our results reproducible and comparable with our teammates.
random_forest_classifier = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    max_depth=10,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42
)

# Why: We train the model on 80% of the patients so it can learn patterns without ever
# seeing the test set, preventing data leakage.
random_forest_classifier.fit(features_train, targets_train)

# Why: We test the model on the remaining 20% of patients it has never seen before,
# giving us an honest measure of real-world performance.
predicted_labels = random_forest_classifier.predict(features_test)